# Bank Marketing: Preprocessing and Baseline Model

## Objective

This notebook prepares the Bank Marketing dataset for machine learning.

The main goals are to:

- define the realistic prediction scenario
- separate features and target
- create training and test sets
- prevent data leakage
- build reproducible preprocessing pipelines
- establish a dummy baseline model

The realistic pre-contact model will exclude `duration` because call duration is not known before the customer is contacted.

In [1]:
from ucimlrepo import fetch_ucirepo

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

bank_marketing = fetch_ucirepo(id=222)

X = bank_marketing.data.features.copy()
y = bank_marketing.data.targets["y"].copy()

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

Feature shape: (45211, 16)
Target shape: (45211,)


## Realistic pre-contact feature set

The intended use case is to predict which customers are likely to subscribe before a marketing call begins.

The `duration` feature is excluded because its value is only known after the call has taken place. Keeping it would make evaluation unrealistically optimistic for this use case.

In [2]:
X_realistic = X.drop(columns=["duration"])

print("Original number of features:", X.shape[1])
print("Realistic number of features:", X_realistic.shape[1])
print("Removed feature: duration")

Original number of features: 16
Realistic number of features: 15
Removed feature: duration


## Target encoding

The target labels are converted from text values to binary numerical values:

- `no` becomes `0`
- `yes` becomes `1`

In [3]:
y_encoded = y.map({
    "no": 0,
    "yes": 1
})

print(y_encoded.value_counts())
print("\nMissing encoded target values:", y_encoded.isna().sum())

y
0    39922
1     5289
Name: count, dtype: int64

Missing encoded target values: 0


## Train-test split

The data is divided into training and test sets.

A stratified split is used so that the proportion of subscribed and non-subscribed customers remains approximately the same in both sets.

The test set is reserved from candidate-model selection and hyperparameter tuning. It is used here only to demonstrate the behaviour of the non-learning dummy baseline, and later for the single final evaluation of the selected trained model..

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X_realistic,
    y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)

print("Training feature shape:", X_train.shape)
print("Test feature shape:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True).round(4))

print("\nTest target distribution:")
print(y_test.value_counts(normalize=True).round(4))

Training feature shape: (36168, 15)
Test feature shape: (9043, 15)

Training target distribution:
y
0    0.883
1    0.117
Name: proportion, dtype: float64

Test target distribution:
y
0    0.883
1    0.117
Name: proportion, dtype: float64


## Feature groups for preprocessing

Numerical and categorical features are identified from the training data.

The two groups require different preprocessing strategies:

- numerical features will be imputed and scaled
- categorical features will be imputed and one-hot encoded

In [5]:
numerical_features = (
    X_train
    .select_dtypes(include="number")
    .columns
    .tolist()
)

categorical_features = (
    X_train
    .select_dtypes(exclude="number")
    .columns
    .tolist()
)

print("Numerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

print("\nNumber of numerical features:", len(numerical_features))
print("Number of categorical features:", len(categorical_features))

Numerical features:
['age', 'balance', 'day_of_week', 'campaign', 'pdays', 'previous']

Categorical features:
['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']

Number of numerical features: 6
Number of categorical features: 9


## Preprocessing pipeline

Separate preprocessing steps are created for numerical and categorical features.

For numerical features:

- missing values are replaced with the median
- values are standardized

For categorical features:

- missing values are replaced with a dedicated `missing` category
- categories are converted using one-hot encoding

All preprocessing steps are combined inside a `ColumnTransformer` so they are learned only from the training data.

In [6]:
numerical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="missing"
            )
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            numerical_pipeline,
            numerical_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ]
)

print(preprocessor)

ColumnTransformer(transformers=[('numerical',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['age', 'balance', 'day_of_week', 'campaign',
                                  'pdays', 'previous']),
                                ('categorical',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(fill_value='missing',
                                                                strategy='constant')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['job', 'marital', 'education', 'default',
                                  'housing', 'loan', 'contact', 'month',
      

In [7]:
dummy_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", DummyClassifier(strategy="most_frequent"))
    ]
)

dummy_pipeline.fit(X_train, y_train)

dummy_predictions = dummy_pipeline.predict(X_test)
dummy_probabilities = dummy_pipeline.predict_proba(X_test)[:, 1]

dummy_results = pd.Series({
    "accuracy": accuracy_score(y_test, dummy_predictions),
    "precision": precision_score(
        y_test,
        dummy_predictions,
        zero_division=0
    ),
    "recall": recall_score(
        y_test,
        dummy_predictions,
        zero_division=0
    ),
    "f1_score": f1_score(
        y_test,
        dummy_predictions,
        zero_division=0
    ),
    "roc_auc": roc_auc_score(
        y_test,
        dummy_probabilities
    ),
    "pr_auc": average_precision_score(
        y_test,
        dummy_probabilities
    )
})

display(dummy_results.to_frame("Dummy classifier"))

,Dummy classifier
accuracy,0.883003
precision,0.000000
recall,0.000000
f1_score,0.000000
roc_auc,0.500000
pr_auc,0.116997


### Baseline interpretation

The dummy classifier predicts the majority class (`no`) for every customer.

It achieves an accuracy of 88.3%, but its precision, recall, and F1-score for subscribers are all zero. Its ROC-AUC is 0.50, which is equivalent to random ranking, while its PR-AUC matches the positive-class prevalence of approximately 11.7%.

This confirms that accuracy alone is misleading for this imbalanced classification problem. Any useful model must outperform this baseline on recall, F1-score, ROC-AUC, and PR-AUC.